# Experiment: Baseline MLP

Calls the training pipeline with a fixed config. Change the config dict to run a new experiment.

In [ ]:
import sys, os

if 'google.colab' in str(get_ipython()):
    REPO = 'diabetes-uci-dataset'
    REPO_URL = 'https://github.com/byambaa0325/diabetes-uci-dataset.git'
    if not os.path.exists(REPO):
        os.system(f'git clone {REPO_URL}')
    os.chdir(REPO)
    os.system('pip install -q -r requirements.txt')
    os.system('pip install -q -e .')
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '../../'))
    if root not in sys.path:
        sys.path.insert(0, root)

In [ ]:
import torch
from src.data.loader import load_raw
from src.data.preprocessor import clean, encode_target
from src.data.features import build_features
from src.models.trainers import run_training
from src.evaluation.metrics import evaluate

In [ ]:
# Data
df_raw, _ = load_raw()
df = encode_target(clean(df_raw))
X, y = build_features(df)  

split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

In [ ]:
# --- Config: pick one block and run ---

# PyTorch MLP
config = {
    'name':          'baseline_mlp',
    'wandb_project': 'applied-ai-coursework',
    'model':         'mlp',            # see src/models/registry.py for all options
    'input_dim':     X_train.shape[1],
    'hidden_dims':   [128, 64],
    'output_dim':    1,
    'dropout':       0.3,
    'lr':            1e-3,
    'epochs':        50,
    'batch_size':    256,
    'X_train': X_train, 'y_train': y_train,
    'X_val':   X_val,   'y_val':   y_val,
}

# Sklearn — swap in instead of the block above
# config = {
#     'name':          'baseline_logreg',
#     'wandb_project': 'applied-ai-coursework',
#     'model':         'logistic_regression',   # or 'random_forest', 'gradient_boosting'
#     'model_params':  {'C': 1.0, 'max_iter': 1000},
#     'X_train': X_train, 'y_train': y_train,
#     'X_val':   X_val,   'y_val':   y_val,
# }

In [ ]:
# Train
run_dir = run_training(config)
print('Artefacts saved to:', run_dir)

In [ ]:
# Evaluate — works for both PyTorch and sklearn
import joblib
from src.models.registry import MODEL_REGISTRY
from src.networks.mlp import MLP

entry = MODEL_REGISTRY[config['model']]

if entry['framework'] == 'torch':
    model = MLP(config['input_dim'], config['hidden_dims'])
    model.load_state_dict(torch.load(run_dir / 'weights.pt'))
else:
    model = joblib.load(run_dir / 'model.joblib')

results = evaluate(model, X_val, y_val)
print(f"ROC-AUC: {results['roc_auc']:.4f}")